# 面试题：不用现成 tokenizer，怎样从零实现 BPE？

## 可以直接复述的回答

BPE 不是按频率直接记住整词，而是从字符等基础符号开始，反复合并全语料中频率最高的相邻符号对。训练阶段最关键的是明确词边界、按词频加权 pair count，并固定并列时的排序规则，否则同一份语料也可能得到不同词表。推理阶段不能重新统计频率，而要严格按训练得到的 merge 顺序逐条应用。衡量 BPE 不能只看词表大小，还应同时看序列压缩率、未登录词可编码性和边界是否被破坏。本题用客服搜索词的离线脱敏样本，从字符基线开始，打印每轮 pair 计数、选择的 merge 和最终编码。最后还会复现“去掉词边界后把两个词粘成一个 token”的错误，并说明生产 tokenizer 还需要 byte fallback、Unicode 规范化和特殊 token 协议。

## 真实案例

数据是结构与电商客服搜索日志一致的离线教学样本：每条记录包含查询词及近似出现次数，共 8 个业务短语；数值为人工脱敏小样本，只用于解释算法，不能外推线上压缩收益。

In [1]:
from collections import Counter  # 导入计数器以统计相邻符号对和词频
from pprint import pprint  # 导入结构化打印函数以展示教学账本
corpus_counts = {"退款流程": 12, "退款申请": 10, "订单退款": 8, "订单查询": 7, "发票申请": 6, "发票查询": 5, "密码重置": 4, "密码修改": 3}  # 构造带真实业务语义的查询词频
evaluation_texts = ["退款申请", "订单退款", "发票查询", "密码修改", "退款查询", "订单申请"]  # 准备包含已见组合与新组合的评估样本
print("输入预览：词频最高的客服查询")  # 输出原始输入标题
pprint([{"查询": word, "频次": count} for word, count in corpus_counts.items()])  # 展示八条查询及其脱敏频次
print("评估样本：", evaluation_texts)  # 展示稍后会在同一数据上比较的文本

输入预览：词频最高的客服查询
[{'查询': '退款流程', '频次': 12},
 {'查询': '退款申请', '频次': 10},
 {'查询': '订单退款', '频次': 8},
 {'查询': '订单查询', '频次': 7},
 {'查询': '发票申请', '频次': 6},
 {'查询': '发票查询', '频次': 5},
 {'查询': '密码重置', '频次': 4},
 {'查询': '密码修改', '频次': 3}]
评估样本： ['退款申请', '订单退款', '发票查询', '密码修改', '退款查询', '订单申请']


## Baseline / 基线：逐字符编码

字符方案永远能编码中文，但不会复用“退款”“订单”等高频片段，因此序列较长。这里先把它作为同数据基线，而不是直接宣称 BPE 更好。

In [2]:
def character_tokens(text):  # 定义最简单的逐字符基线
    return list(text)  # 把每个 Unicode 字符直接当作一个 token
baseline_rows = []  # 创建逐样本基线结果表
for text in evaluation_texts:  # 遍历同一组评估文本
    tokens = character_tokens(text)  # 执行字符级切分
    baseline_rows.append({"文本": text, "tokens": tokens, "长度": len(tokens)})  # 保存可读 token 与长度
print("字符 Baseline：每条文本都能编码，但常用片段没有压缩")  # 输出基线说明
pprint(baseline_rows)  # 展示逐样本基线结果

字符 Baseline：每条文本都能编码，但常用片段没有压缩
[{'tokens': ['退', '款', '申', '请'], '文本': '退款申请', '长度': 4},
 {'tokens': ['订', '单', '退', '款'], '文本': '订单退款', '长度': 4},
 {'tokens': ['发', '票', '查', '询'], '文本': '发票查询', '长度': 4},
 {'tokens': ['密', '码', '修', '改'], '文本': '密码修改', '长度': 4},
 {'tokens': ['退', '款', '查', '询'], '文本': '退款查询', '长度': 4},
 {'tokens': ['订', '单', '申', '请'], '文本': '订单申请', '长度': 4}]


## 手写核心：统计 pair、合并 pair、保留词边界

每个词末尾加入边界符，词频作为统计权重。pair 只在单个词的符号序列内部统计，因此不会跨查询或跨词误合并。

In [3]:
def build_vocabulary(word_counts):  # 把原始词频转换为带边界的符号序列
    return {tuple(list(word) + ["</w>"]): count for word, count in word_counts.items()}  # 为每个词显式追加词尾标记
def count_pairs(vocabulary):  # 定义按词频加权的相邻 pair 统计
    pair_counts = Counter()  # 创建 pair 频次计数器
    for symbols, frequency in vocabulary.items():  # 遍历每个符号序列及其出现次数
        for index in range(len(symbols) - 1):  # 只查看当前词内部的相邻位置
            pair_counts[(symbols[index], symbols[index + 1])] += frequency  # 按词频累加当前 pair
    return pair_counts  # 返回完整 pair 频次表
def merge_pair(vocabulary, target_pair):  # 定义一次确定性的 pair 合并
    merged_vocabulary = {}  # 创建合并后的新词表
    for symbols, frequency in vocabulary.items():  # 逐词应用同一条 merge 规则
        merged_symbols = []  # 收集当前词合并后的符号
        cursor = 0  # 从词首开始扫描
        while cursor < len(symbols):  # 持续扫描直到词尾
            can_merge = cursor + 1 < len(symbols) and (symbols[cursor], symbols[cursor + 1]) == target_pair  # 判断当前位置是否命中目标 pair
            if can_merge:  # 命中规则时把两个 token 拼成一个新 token
                merged_symbols.append(symbols[cursor] + symbols[cursor + 1])  # 追加合并后的 token
                cursor += 2  # 跳过已经消费的两个旧 token
            else:  # 未命中规则时保留原 token
                merged_symbols.append(symbols[cursor])  # 追加当前位置的原始 token
                cursor += 1  # 向后移动一个位置
        merged_vocabulary[tuple(merged_symbols)] = frequency  # 保存当前词的新符号序列及词频
    return merged_vocabulary  # 返回完成一次 merge 的词表
initial_vocabulary = build_vocabulary(corpus_counts)  # 构造带词边界的初始字符词表
print("初始符号序列示例：")  # 输出核心实现的输入状态
pprint(list(initial_vocabulary.items())[:4])  # 展示前四个词及其词频权重

初始符号序列示例：
[(('退', '款', '流', '程', '</w>'), 12),
 (('退', '款', '申', '请', '</w>'), 10),
 (('订', '单', '退', '款', '</w>'), 8),
 (('订', '单', '查', '询', '</w>'), 7)]


In [4]:
working_vocabulary = initial_vocabulary  # 复制初始状态供迭代训练
merge_rules = []  # 按顺序保存训练得到的 merge 规则
merge_ledger = []  # 保存每轮最关键的 pair 统计账本
for round_index in range(8):  # 执行八轮小规模 BPE 训练
    pair_counts = count_pairs(working_vocabulary)  # 统计当前符号状态下的加权 pair 频次
    ranked_pairs = sorted(pair_counts.items(), key=lambda item: (-item[1], item[0]))  # 用频次降序和字典序打破并列
    best_pair, best_count = ranked_pairs[0]  # 取本轮频次最高的 pair
    merge_rules.append(best_pair)  # 记录规则以便推理阶段按同样顺序执行
    merge_ledger.append({"轮次": round_index + 1, "top3_pair": ranked_pairs[:3], "选择": best_pair, "加权频次": best_count})  # 保存本轮候选和最终选择
    working_vocabulary = merge_pair(working_vocabulary, best_pair)  # 把本轮规则应用到全部词
print("BPE 每轮 pair 账本：")  # 输出训练过程标题
pprint(merge_ledger)  # 展示每轮 top pair、频次与实际选择
print("最终 merge 顺序：", merge_rules)  # 展示推理必须复用的有序规则

BPE 每轮 pair 账本：
[{'top3_pair': [(('退', '款'), 30), (('申', '请'), 16), (('请', '</w>'), 16)],
  '加权频次': 30,
  '轮次': 1,
  '选择': ('退', '款')},
 {'top3_pair': [(('申', '请'), 16), (('请', '</w>'), 16), (('订', '单'), 15)],
  '加权频次': 16,
  '轮次': 2,
  '选择': ('申', '请')},
 {'top3_pair': [(('申请', '</w>'), 16), (('订', '单'), 15), (('查', '询'), 12)],
  '加权频次': 16,
  '轮次': 3,
  '选择': ('申请', '</w>')},
 {'top3_pair': [(('订', '单'), 15), (('查', '询'), 12), (('流', '程'), 12)],
  '加权频次': 15,
  '轮次': 4,
  '选择': ('订', '单')},
 {'top3_pair': [(('查', '询'), 12), (('流', '程'), 12), (('程', '</w>'), 12)],
  '加权频次': 12,
  '轮次': 5,
  '选择': ('查', '询')},
 {'top3_pair': [(('查询', '</w>'), 12), (('流', '程'), 12), (('程', '</w>'), 12)],
  '加权频次': 12,
  '轮次': 6,
  '选择': ('查询', '</w>')},
 {'top3_pair': [(('流', '程'), 12), (('程', '</w>'), 12), (('退款', '流'), 12)],
  '加权频次': 12,
  '轮次': 7,
  '选择': ('流', '程')},
 {'top3_pair': [(('流程', '</w>'), 12), (('退款', '流程'), 12), (('发', '票'), 11)],
  '加权频次': 12,
  '轮次': 8,
  '选择': ('流程', '</w>')}]
最终 mer

## 结果表与结果解读

推理严格复用训练得到的规则，不再查看评估集频率。已见高频片段应比字符基线更短；新组合仍可退回字符，因此不会出现整词词典那样的完全未登录问题。压缩率只是教学语料上的观测值，不代表线上真实分布。

In [5]:
def encode_word(word, rules):  # 定义按训练顺序编码单个词的推理函数
    symbols = list(word) + ["</w>"]  # 从字符和显式词尾标记开始
    for target_pair in rules:  # 严格按训练得到的规则顺序处理
        merged_symbols = []  # 收集当前规则处理后的 token
        cursor = 0  # 从序列起点扫描
        while cursor < len(symbols):  # 扫描当前全部 token
            can_merge = cursor + 1 < len(symbols) and (symbols[cursor], symbols[cursor + 1]) == target_pair  # 判断当前位置是否匹配当前规则
            if can_merge:  # 匹配时合并相邻 token
                merged_symbols.append(symbols[cursor] + symbols[cursor + 1])  # 写入合并 token
                cursor += 2  # 同时消费两个输入 token
            else:  # 不匹配时保留当前 token
                merged_symbols.append(symbols[cursor])  # 写入原 token
                cursor += 1  # 消费一个输入 token
        symbols = merged_symbols  # 把本轮结果作为下一轮输入
    return symbols  # 返回含词尾信息的最终 token 序列
comparison_rows = []  # 创建字符基线与 BPE 的逐样本对照表
for text in evaluation_texts:  # 遍历六条相同评估文本
    char_tokens = character_tokens(text)  # 获取字符基线结果
    bpe_tokens = encode_word(text, merge_rules)  # 获取手写 BPE 结果
    comparison_rows.append({"文本": text, "字符长度": len(char_tokens), "BPE长度": len(bpe_tokens), "BPE tokens": bpe_tokens, "压缩率": round(1 - len(bpe_tokens) / (len(char_tokens) + 1), 3)})  # 记录边界在内的公平长度对照
print("逐样本编码结果：")  # 输出结果表标题
pprint(comparison_rows)  # 展示每条文本的 token 与压缩情况

逐样本编码结果：
[{'BPE tokens': ['退款', '申请</w>'],
  'BPE长度': 2,
  '压缩率': 0.6,
  '字符长度': 4,
  '文本': '退款申请'},
 {'BPE tokens': ['订单', '退款', '</w>'],
  'BPE长度': 3,
  '压缩率': 0.4,
  '字符长度': 4,
  '文本': '订单退款'},
 {'BPE tokens': ['发', '票', '查询</w>'],
  'BPE长度': 3,
  '压缩率': 0.4,
  '字符长度': 4,
  '文本': '发票查询'},
 {'BPE tokens': ['密', '码', '修', '改', '</w>'],
  'BPE长度': 5,
  '压缩率': 0.0,
  '字符长度': 4,
  '文本': '密码修改'},
 {'BPE tokens': ['退款', '查询</w>'],
  'BPE长度': 2,
  '压缩率': 0.6,
  '字符长度': 4,
  '文本': '退款查询'},
 {'BPE tokens': ['订单', '申请</w>'],
  'BPE长度': 2,
  '压缩率': 0.6,
  '字符长度': 4,
  '文本': '订单申请'}]


## 失败案例：删除词边界会学到跨词短语

如果训练前直接删除空格，把“退款 申请”当成连续 token 流，高频相邻词可能被错误合并成“退款申请”。这样既混淆词内子词与跨词短语，也降低 token 在别的上下文中的复用性。修正方式是按词独立统计并保留边界；句子级 tokenizer 还要明确空格或 byte 的编码协议。

In [6]:
def unsafe_phrase_merge(tokens, target_pair):  # 定义忽略词边界的错误合并器
    if tuple(tokens[:2]) == target_pair:  # 检查前两个词是否恰好等于高频短语 pair
        return [tokens[0] + tokens[1]] + tokens[2:]  # 错误地把两个独立词粘成一个 token
    return tokens  # 未命中时返回原 token
sentence_words = ["退款", "申请"]  # 构造包含两个独立业务词的句子
unsafe_tokens = unsafe_phrase_merge(sentence_words, ("退款", "申请"))  # 模拟删除边界后的跨词 merge
safe_tokens = [encode_word(word, merge_rules) for word in sentence_words]  # 对每个词独立编码并保留边界
print("失败案例（无边界）：", unsafe_tokens)  # 展示错误的跨词 token
print("修正后（逐词带边界）：", safe_tokens)  # 展示不会跨词合并的结果

失败案例（无边界）： ['退款申请']
修正后（逐词带边界）： [['退款', '</w>'], ['申请</w>']]


## 生产差距

真实 tokenizer 通常从 UTF-8 byte 或规范化后的字符开始，还要处理 emoji、组合字符、特殊 token、词表版本兼容和增量数据污染。大语料训练需要高效更新 pair 统计而不是每轮全量扫描；部署侧必须把词表、merge 表、规范化规则一起版本化，并用多语言、恶意 Unicode 与往返解码测试守护。

In [7]:
assert len(corpus_counts) >= 8  # 验证真实案例至少包含八个业务查询
assert len(merge_rules) == 8  # 验证训练确实完成八轮确定性合并
assert merge_rules[0] == ("退", "款")  # 验证最高频业务片段首先被合并
assert all(row["BPE长度"] <= row["字符长度"] + 1 for row in comparison_rows)  # 验证 BPE 不比含词尾的字符表示更长
assert unsafe_tokens == ["退款申请"]  # 验证失败案例确实发生跨词粘连
assert len(safe_tokens) == 2  # 验证修正方案保留两个独立词边界
print("最小回归测试通过：BPE 训练、编码与边界保护均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：BPE 训练、编码与边界保护均满足预期
